# Section 7 – Draw the card and share it with other services
**STUDENT notebook: complete the TODO cells**

| | |
|---|---|
| Connects to | Topic 3 – FastAPI (binary responses, routers) and the whole lab |
| Estimated time | 75 minutes |
| Produces | `app/card.py`, `app/cards_router.py`, `app/main_cards.py` and PNG cards in `outputs/` |

**Goal:** Turn data + predictions into an image (Pillow), serve the PNG through the API, and prove that a second, independent program can consume the product.

### How to work in this notebook
- 🟦 **Worked sample** cells run as they are. Read them, run them, change them.
- 🟨 **Your turn** cells contain `# TODO` and `...`. Replace `...` with your code.
- ✅ **Check** cells print `PASS` / `FAIL`. Do not move on while you see a `FAIL`.
- Stuck? Open the same file in the `solution/` folder, but try first.

## Step 0 – Setup

In [ ]:
import importlib.util
import subprocess
import sys

# 1) Make sure the libraries exist in THIS kernel (installs only what is missing)
REQUIRED = {"requests": "requests", "pandas": "pandas", "numpy": "numpy",
            "scikit-learn": "sklearn", "matplotlib": "matplotlib", "pillow": "PIL",
            "joblib": "joblib", "fastapi": "fastapi", "uvicorn": "uvicorn",
            "httpx": "httpx", "pydantic": "pydantic"}
missing = [pkg for pkg, mod in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("Installed:", missing, "-> restart the kernel if an import below fails")
else:
    print("All dependencies are available")

# 2) Corporate networks with HTTPS inspection: use the Windows certificate store if available
try:
    import truststore
    truststore.inject_into_ssl()
    print("truststore active (Windows certificate store)")
except ImportError:
    pass

# 3) Lab folders (this notebook lives in student/ or solution/, so the lab root is one level up)
from pathlib import Path
import json

ROOT = Path.cwd().parent
DATA, MODELS, OUT = ROOT / "data", ROOT / "models", ROOT / "outputs"
for folder in (DATA, MODELS, OUT):
    folder.mkdir(exist_ok=True)


def check(name, ok, detail=""):
    """Self-check helper: prints PASS / FAIL."""
    print(("PASS" if ok else "FAIL"), "-", name, ("-> " + str(detail)) if detail else "")


print("Lab root:", ROOT)

## The final product
```
GET  /v1/characters/{id}/card.png     card of a real character (artwork + stats + prediction)
POST /v1/cards                        card of YOUR OWN character (JSON in → PNG out)
```

A card shows:

| Zone | Content | Comes from |
|---|---|---|
| Background color | the character's type | dataset |
| Border color | rarity: grey / blue / gold | **classification model** |
| Picture | official artwork | external image service (cached) |
| Bars | the 6 stats | dataset or request |
| Footer | **predicted power** vs real power | **regression model** |

Requirement: Sections 5 and 6 done (`app/` with `main.py`). Otherwise copy `solution/app/` into `student/app/`.

## Step 1 – Pillow in 2 minutes
Pillow draws on an in-memory canvas: `Image.new` → `ImageDraw` → `save`. An API returns an image as **bytes**
with the right `Content-Type` (`image/png`).

🟦 **Worked sample**:

In [ ]:
from io import BytesIO
from PIL import Image, ImageDraw, ImageFont
from IPython.display import Image as ShowImage, display

img = Image.new("RGB", (320, 120), (30, 60, 120))
draw = ImageDraw.Draw(img)
draw.rounded_rectangle((8, 8, 311, 111), radius=18, outline=(240, 180, 0), width=5)
draw.text((24, 30), "HELLO CARD", fill="white", font=ImageFont.load_default(32))
draw.rectangle((24, 80, 24 + 200, 94), fill=(0, 160, 80))          # a "stat bar" 200 px wide

buffer = BytesIO()
img.save(buffer, format="PNG")                  # image -> PNG bytes (this is what an API sends)
print("PNG size:", len(buffer.getvalue()), "bytes | starts with:", buffer.getvalue()[:4])
display(ShowImage(buffer.getvalue()))

## Step 2 – Download the artwork (and cache it)
Artwork lives on another host (images are not part of the JSON). We download it **once**, keep it in `data/images/`
and the next request reads the file. If the download fails, the card must still be drawn (a placeholder).

🟨 You write the core of `load_artwork`.

In [ ]:
Path("app").mkdir(exist_ok=True)

In [ ]:
%%writefile app/card.py
"""Draws the collectible card (Pillow) from a character, its prediction and its artwork."""
from io import BytesIO
from pathlib import Path
from typing import Literal, Optional

import requests
from PIL import Image, ImageDraw, ImageFont
from pydantic import BaseModel, Field

from .config import DATA_DIR
from .schemas import CharacterStats

CARD_W, CARD_H = 600, 840
TYPE_COLORS = {
    "normal": (168, 167, 122), "fire": (240, 128, 48), "water": (104, 144, 240), "electric": (248, 208, 48),
    "grass": (120, 200, 80), "ice": (152, 216, 216), "fighting": (192, 48, 40), "poison": (160, 64, 160),
    "ground": (224, 192, 104), "flying": (168, 144, 240), "psychic": (248, 88, 136), "bug": (168, 184, 32),
    "rock": (184, 160, 56), "ghost": (112, 88, 152), "dragon": (112, 56, 248), "dark": (112, 88, 72),
    "steel": (184, 184, 208), "fairy": (238, 153, 172),
}
RARITY_COLORS = {"common": (150, 150, 150), "rare": (50, 110, 220), "legendary": (240, 180, 0)}
STAT_LABELS = [("hp", "HP"), ("attack", "ATK"), ("defense", "DEF"),
               ("special_attack", "SP.ATK"), ("special_defense", "SP.DEF"), ("speed", "SPD")]
TypeName = Literal["normal", "fire", "water", "electric", "grass", "ice", "fighting", "poison", "ground",
                   "flying", "psychic", "bug", "rock", "ghost", "dragon", "dark", "steel", "fairy"]


class CardRequest(BaseModel):
    """Body of POST /v1/cards: a custom character."""
    name: str = Field(min_length=1, max_length=20)
    type1: TypeName = "normal"
    stats: CharacterStats


def get_font(size: int, bold: bool = False):
    names = ["arialbd.ttf", "DejaVuSans-Bold.ttf"] if bold else ["arial.ttf", "DejaVuSans.ttf"]
    for name in names:
        try:
            return ImageFont.truetype(name, size)
        except OSError:
            continue
    return ImageFont.load_default(size)


def lighten(color, amount=0.55):
    return tuple(int(c + (255 - c) * amount) for c in color)


def load_artwork(url: Optional[str], character_id: int) -> Optional[Image.Image]:
    """Return the artwork as an RGBA image, using a local cache. None if it cannot be loaded."""
    if not url:
        return None
    cache = DATA_DIR / "images" / f"{character_id}.png"
    try:
        # TODO: if the cached file does not exist: GET the url (timeout 15), raise for bad status and write the bytes to `cache`
        # HINT: cache.parent.mkdir(parents=True, exist_ok=True); response.content holds the bytes
        ...  # <- write your code here
        return Image.open(cache).convert("RGBA")
    except (requests.exceptions.RequestException, OSError):
        return None


def build_card(name: str, type1: str, stats: dict, prediction: dict,
               actual_power: Optional[int] = None, artwork: Optional[Image.Image] = None) -> Image.Image:
    """Draw the card and return a Pillow image."""
    type_color = TYPE_COLORS.get(type1, TYPE_COLORS["normal"])
    # TODO: border color from the rarity of the prediction (RARITY_COLORS)
    ...  # <- write your code here

    img = Image.new("RGB", (CARD_W, CARD_H), rarity_color)
    draw = ImageDraw.Draw(img)
    draw.rounded_rectangle((0, 0, CARD_W - 1, CARD_H - 1), radius=36, outline=(40, 40, 40), width=3)
    draw.rounded_rectangle((18, 18, CARD_W - 19, CARD_H - 19), radius=26, fill=lighten(type_color))

    # header
    draw.text((40, 50), name.capitalize(), fill=(30, 30, 30), font=get_font(46, bold=True), anchor="lm")
    draw.rounded_rectangle((440, 30, 566, 70), radius=14, fill=type_color)
    draw.text((503, 50), type1.upper(), fill="white", font=get_font(20, bold=True), anchor="mm")

    # picture frame
    draw.rounded_rectangle((70, 95, 530, 495), radius=24, fill=(255, 255, 255), outline=type_color, width=5)
    if artwork is not None:
        art = artwork.copy()
        art.thumbnail((400, 360))
        img.paste(art, (300 - art.width // 2, 295 - art.height // 2), art)
    else:
        draw.ellipse((200, 195, 400, 395), fill=type_color)
        draw.text((300, 295), name[:1].upper(), fill="white", font=get_font(110, bold=True), anchor="mm")

    # stat bars (maximum value 255 = full bar)
    y = 515
    for key, label in STAT_LABELS:
        value = stats[key]
        draw.text((40, y + 14), label, fill=(30, 30, 30), font=get_font(20, bold=True), anchor="lm")
        draw.rounded_rectangle((150, y + 3, 480, y + 25), radius=8, fill=(225, 225, 225))
        # TODO: filled part of the bar: width proportional to value / 255 (at least 12 px so that it is visible)
        # HINT: bar_w = max(12, int(330 * value / 255))
        ...  # <- write your code here
        draw.rounded_rectangle((150, y + 3, 150 + bar_w, y + 25), radius=8, fill=type_color)
        draw.text((565, y + 14), str(value), fill=(30, 30, 30), font=get_font(20), anchor="rm")
        y += 32

    # footer: prediction vs reality + rarity
    draw.rounded_rectangle((36, 718, CARD_W - 37, 806), radius=18, fill=(35, 35, 45))
    draw.text((56, 745), f"PREDICTED POWER  {prediction['predicted_power']:.0f}", fill="white",
              font=get_font(26, bold=True), anchor="lm")
    # TODO: second footer line: if actual_power is known, show "actual X  (error +/-N)"; otherwise "custom character"
    ...  # <- write your code here
    draw.text((56, 782), second, fill=(200, 200, 210), font=get_font(20), anchor="lm")
    draw.text((CARD_W - 56, 745), prediction["rarity"].upper(), fill=rarity_color,
              font=get_font(26, bold=True), anchor="rm")
    draw.text((CARD_W - 56, 782), f"{prediction['legendary_probability'] * 100:.0f}% legendary",
              fill=(200, 200, 210), font=get_font(20), anchor="rm")
    return img


def card_to_png(img: Image.Image) -> bytes:
    buffer = BytesIO()
    img.save(buffer, format="PNG")
    return buffer.getvalue()

## Step 3 – Look at a card before serving it
Import the module and draw one card directly from the table, with no API involved.

In [ ]:
import sys
import time
import threading
import requests
import uvicorn
import pandas as pd
from IPython.display import Image as ShowImage, display

sys.path.insert(0, str(Path.cwd()))
for name in [m for m in sys.modules if m == "app" or m.startswith("app.")]:
    del sys.modules[name]

from app.model_service import ModelService
from app.card import build_card, card_to_png, load_artwork

service = ModelService()
clean = pd.read_csv(DATA / "pokemon_clean.csv")


def card_for(character_id):
    row = clean[clean["id"] == character_id].iloc[0]
    stats = {f: int(row[f]) for f in service.features}
    prediction = service.predict(stats)
    art = load_artwork(row["artwork_url"], int(row["id"]))
    return build_card(row["name"], row["type1"], stats, prediction, int(row["base_experience"]), art), art


img, art = card_for(25)
print("artwork downloaded:", art is not None)
png = card_to_png(img)
(OUT / "card_pikachu_direct.png").write_bytes(png)
display(ShowImage(png, width=300))
check("PNG signature", png[:8] == b"\x89PNG\r\n\x1a\n")
check("card size is 600x840", img.size == (600, 840), img.size)

## Step 4 – Serve cards with a router
A big API is split into **routers**: each file groups related routes. `main.py` stays small; the new file adds the card routes.

```
app/main.py          characters + predictions  (Section 6)
app/cards_router.py  the two card routes       (this section)
app/main_cards.py    main + cards_router together  → the final app
```
Note the important detail: the route returns `Response(content=bytes, media_type="image/png")`, **not JSON**.

In [ ]:
%%writefile app/cards_router.py
"""Card routes: the same API, now able to return images."""
from fastapi import APIRouter, Response

from .card import CardRequest, build_card, card_to_png, load_artwork
from .main import get_row_or_404, state

router = APIRouter(prefix="/v1", tags=["cards"])

PNG_HEADERS = {"Cache-Control": "public, max-age=3600"}


@router.get("/characters/{character_id}/card.png", response_class=Response,
            responses={200: {"content": {"image/png": {}}}})
def character_card(character_id: int):
    row = get_row_or_404(character_id)
    service = state["service"]
    stats = {f: int(row[f]) for f in service.features}
    prediction = service.predict(stats)
    artwork = load_artwork(row["artwork_url"], int(row["id"]))
    img = build_card(row["name"], row["type1"], stats, prediction, int(row["base_experience"]), artwork)
    # TODO: return the PNG bytes with media_type "image/png" and the PNG_HEADERS
    # HINT: Response(content=card_to_png(img), media_type="image/png", headers=PNG_HEADERS)
    ...  # <- write your code here


@router.post("/cards", response_class=Response, responses={200: {"content": {"image/png": {}}}})
def custom_card(request: CardRequest):
    stats = request.stats.model_dump()
    # TODO: predict for the custom stats, draw the card WITHOUT artwork and actual power, and return the PNG
    ...  # <- write your code here

In [ ]:
%%writefile app/main_cards.py
"""FINAL APPLICATION: characters + predictions + cards."""
from .cards_router import router
from .main import app

app.include_router(router)

## Step 5 – Test the final app

In [ ]:
for name in [m for m in sys.modules if m == "app" or m.startswith("app.")]:
    del sys.modules[name]

from fastapi.testclient import TestClient
from app.main_cards import app

with TestClient(app) as client:
    r = client.get("/v1/characters/150/card.png")
    check("card route -> 200 image/png", r.status_code == 200 and r.headers["content-type"] == "image/png",
          r.headers.get("content-type"))
    check("body is a PNG", r.content[:8] == b"\x89PNG\r\n\x1a\n")
    check("cache header present", "max-age" in r.headers.get("cache-control", ""))

    r404 = client.get("/v1/characters/999999/card.png")
    check("unknown character -> 404 JSON error", r404.status_code == 404 and r404.json()["error"]["code"] == "CHARACTER_NOT_FOUND")

    custom = {"name": "Luis", "type1": "dragon",
              "stats": {"hp": 90, "attack": 120, "defense": 80, "special_attack": 100,
                        "special_defense": 80, "speed": 95, "height": 18, "weight": 1500}}
    r = client.post("/v1/cards", json=custom)
    check("custom card -> 200 image/png", r.status_code == 200 and r.headers["content-type"] == "image/png")
    (OUT / "card_custom_luis.png").write_bytes(r.content)

    bad = {**custom, "type1": "banana"}
    check("unknown type -> 422", client.post("/v1/cards", json=bad).status_code == 422)

    paths = client.get("/openapi.json").json()["paths"]
    check("cards are documented in OpenAPI", "/v1/cards" in paths and "/v1/characters/{character_id}/card.png" in paths)

## Step 6 – Another program consumes the product
This is the point of the whole lab: **applications sharing data**. We start our API as a real server, and then a
completely separate "consumer" (it knows only URLs and JSON, nothing about models or Pillow) asks for the legendary
characters and downloads their cards.

In [ ]:
PORT = 8780
BASE_URL = f"http://127.0.0.1:{PORT}"
server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=PORT, log_level="warning"))
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
for _ in range(100):
    try:
        requests.get(BASE_URL, timeout=0.5)
        break
    except requests.exceptions.RequestException:
        time.sleep(0.1)
print("API running at", BASE_URL)

In [ ]:
# ---------- the CONSUMER: only knows HTTP ----------
(OUT / "consumer").mkdir(exist_ok=True)

# TODO: ask the API for 3 legendary characters (query parameters legendary=true and limit=3) and keep the list in `items`
# HINT: requests.get(f"{BASE_URL}/v1/characters", params={...}, timeout=10).json()["items"]
...  # <- write your code here

saved = []
for item in items:
    # TODO: download the card PNG of each item and save it as outputs/consumer/<name>.png
    # HINT: the URL is f"{BASE_URL}/v1/characters/{item['id']}/card.png"
    ...  # <- write your code here
    saved.append(path)
    print("saved", path.name, len(r.content), "bytes")

check("3 cards downloaded by the consumer", len(saved) == 3 and all(p.exists() for p in saved))
for p in saved:
    display(ShowImage(filename=str(p), width=220))

In [ ]:
server.should_exit = True
thread.join(timeout=5)
print("server stopped:", not thread.is_alive())

## Step 7 – Contact sheet: compare predictions at a glance
We draw four characters side by side. Look at the borders (rarity), the footers (predicted vs actual) and discuss the model.

In [ ]:
from PIL import Image

ids = [25, 129, 143, 150]           # pikachu, magikarp, snorlax, mewtwo
sheet = Image.new("RGB", (4 * 300, 420), "white")
for k, cid in enumerate(ids):
    card, _ = card_for(cid)
    sheet.paste(card.resize((300, 420)), (k * 300, 0))
sheet.save(OUT / "contact_sheet.png")
display(ShowImage(filename=str(OUT / "contact_sheet.png")))
check("contact sheet saved", (OUT / "contact_sheet.png").exists())

## Final checklist
| ✔ | Item |
|---|---|
| ☐ | Section 1: `pokemon_raw.csv` collected from the API |
| ☐ | Section 2: clean table + feature ranges |
| ☐ | Section 3: regression models compared, best saved |
| ☐ | Section 4: classification models compared, best saved |
| ☐ | Section 5: Pydantic schemas + `ModelService` |
| ☐ | Section 6: API with tests, standard errors, OpenAPI |
| ☐ | Section 7: card PNG served by the API and downloaded by a separate consumer |

## Reflection
1. Draw the architecture of the lab (boxes and arrows) from memory.
2. Which three formats traveled between programs? (JSON, PNG, joblib file?) Which ones cross the network?
3. If the artwork host is down, what does the card show? Where is that decision in the code?
4. How would an **AI assistant** use this API? (That is Topics 11–14: MCP.)

### Where to go next
- **UI (Topic 7):** a Streamlit page with sliders for the 8 stats that calls `POST /v1/cards` and shows the image.
- **MCP (Topics 11–14):** wrap `create_prediction` and `card` as MCP tools so an assistant can say "make me a card for a fast, fragile dragon".
- **Docker (Topic 17):** package the API and the models in one container.